# Lab 05: Grover's Algorithm

**Goal:** Implement Grover's search algorithm to find a marked item in an unsorted database.

---

In [ ]:
# Install packages and force Python to recognize them
import subprocess
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "qiskit", "qiskit-aer", "pylatexenc", "matplotlib", "-q"])

import importlib
importlib.invalidate_caches()
import pylatexenc

from qiskit.utils import optionals
if hasattr(optionals, 'HAS_PYLATEXENC'):
    optionals.HAS_PYLATEXENC._is_available = True

from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram
import numpy as np
print("Ready!")

## Understanding the Problem

**Scenario:** You have a database of N items. One item is "marked" as the solution.
- Classical: Check items one by one → O(N) checks
- Grover's: Use quantum interference → O(√N) checks

We'll search a 2-qubit space (4 items) and find the marked item.

## Exercise 1: The Oracle

The oracle "marks" the solution by flipping its phase.

In [ ]:
def oracle_11(qc):
    """Oracle that marks |11⟩ as the solution"""
    qc.cz(0, 1)  # Controlled-Z flips phase of |11⟩

def oracle_00(qc):
    """Oracle that marks |00⟩ as the solution"""
    qc.x(0)
    qc.x(1)
    qc.cz(0, 1)
    qc.x(0)
    qc.x(1)

def oracle_10(qc):
    """Oracle that marks |10⟩ as the solution"""
    qc.x(1)
    qc.cz(0, 1)
    qc.x(1)

## Exercise 2: The Diffusion Operator

The diffusion operator amplifies the marked state.

In [ ]:
def diffusion(qc):
    """Grover diffusion operator for 2 qubits"""
    # Apply H gates
    qc.h(0)
    qc.h(1)
    
    # Apply X gates
    qc.x(0)
    qc.x(1)
    
    # Controlled-Z
    qc.cz(0, 1)
    
    # Apply X gates
    qc.x(0)
    qc.x(1)
    
    # Apply H gates
    qc.h(0)
    qc.h(1)

## Exercise 3: Complete Grover's Algorithm

In [ ]:
def grover_2qubit(oracle_func, iterations=1):
    """Complete Grover's algorithm for 2 qubits"""
    qc = QuantumCircuit(2, 2)
    
    # Step 1: Initialize - put all qubits in superposition
    qc.h(0)
    qc.h(1)
    qc.barrier()
    
    # Step 2: Apply Grover iteration(s)
    for _ in range(iterations):
        # Apply oracle
        oracle_func(qc)
        qc.barrier()
        
        # Apply diffusion
        diffusion(qc)
        qc.barrier()
    
    # Step 3: Measure
    qc.measure([0, 1], [0, 1])
    
    return qc

In [ ]:
# Ensure pylatexenc is recognized
from qiskit.utils import optionals
if hasattr(optionals, 'HAS_PYLATEXENC'):
    optionals.HAS_PYLATEXENC._is_available = True

# Search for |11⟩
qc = grover_2qubit(oracle_11, iterations=1)
qc.draw('mpl')

In [ ]:
simulator = AerSimulator()
result = simulator.run(qc, shots=1000).result()
print("Searching for |11⟩:")
print("Results:", result.get_counts())
plot_histogram(result.get_counts())

**Success!** The state |11⟩ now has very high probability (close to 100%).

## Exercise 4: Search for Different States

In [ ]:
# Search for |00⟩
qc = grover_2qubit(oracle_00, iterations=1)
result = simulator.run(qc, shots=1000).result()
print("Searching for |00⟩:")
plot_histogram(result.get_counts())

In [ ]:
# Search for |10⟩
qc = grover_2qubit(oracle_10, iterations=1)
result = simulator.run(qc, shots=1000).result()
print("Searching for |10⟩:")
plot_histogram(result.get_counts())

## Exercise 5: What Happens with Too Many Iterations?

In [ ]:
# Try different numbers of iterations
print("Effect of iteration count:")
print("-" * 40)

for iters in range(0, 5):
    qc = grover_2qubit(oracle_11, iterations=iters)
    result = simulator.run(qc, shots=1000).result()
    counts = result.get_counts()
    p_correct = counts.get('11', 0) / 1000
    print(f"Iterations: {iters}, P(correct): {p_correct:.2f}")

**Key Insight:** Too many iterations "overshoot" and reduce the probability!

For N=4 items, 1 iteration is optimal. The formula is approximately π√N/4 iterations.

## Exercise 6: 3-Qubit Grover (8 items)

In [ ]:
def oracle_111(qc):
    """Oracle that marks |111⟩"""
    qc.ccz(0, 1, 2)

def diffusion_3qubit(qc):
    """Diffusion for 3 qubits"""
    for i in range(3):
        qc.h(i)
        qc.x(i)
    
    qc.ccz(0, 1, 2)
    
    for i in range(3):
        qc.x(i)
        qc.h(i)

def grover_3qubit(iterations=2):
    qc = QuantumCircuit(3, 3)
    
    # Initialize
    for i in range(3):
        qc.h(i)
    
    # Grover iterations
    for _ in range(iterations):
        oracle_111(qc)
        diffusion_3qubit(qc)
    
    qc.measure([0, 1, 2], [0, 1, 2])
    return qc

# Optimal iterations for N=8: π√8/4 ≈ 2
qc = grover_3qubit(iterations=2)
result = simulator.run(qc, shots=1000).result()
print("3-qubit Grover searching for |111⟩:")
plot_histogram(result.get_counts())

## Challenge: Create Your Own Oracle

Modify the oracle to search for a different state (e.g., |101⟩).

In [ ]:
# Your code here!
def oracle_101(qc):
    """Oracle that marks |101⟩"""
    qc.x(1)  # Flip qubit 1 so |101⟩ becomes |111⟩
    qc.ccz(0, 1, 2)
    qc.x(1)  # Unflip

def grover_custom(oracle_func, iterations=2):
    qc = QuantumCircuit(3, 3)
    for i in range(3):
        qc.h(i)
    for _ in range(iterations):
        oracle_func(qc)
        diffusion_3qubit(qc)
    qc.measure([0, 1, 2], [0, 1, 2])
    return qc

qc = grover_custom(oracle_101, iterations=2)
result = simulator.run(qc, shots=1000).result()
print("Searching for |101⟩:")
plot_histogram(result.get_counts())